# Python Notebook 2: Classification Modelling & Hyperparameter Tuning
**Author:** Mihiri Pabasara | **Student ID:** [Your Student ID]
**Module:** 5DATA002W.2 – Machine Learning & Data Mining
**Peer Reviewer:** [Peer Reviewer Full Name] | **Review Date:** [Date of Review]

> Code reused from **Seminar 3 (Build and Evaluate Predictive Models)**, **Seminar 4 (kNN & Naïve Bayes)**, and **Code Reuse Session 2** as noted per cell.

**Models Built:**
- Naïve Bayes (NB) — Parametric
- Logistic Regression (LR) — Parametric
- K-Nearest Neighbours (KNN) — Non-Parametric

**Success Criteria (from coursework brief):**
> *"The model should aim to predict as many 'Reject' status for clients as possible to decrease the risk of future defaulted loan payments. However, the model should demonstrate that its high 'Reject' prediction rate is mainly due to a larger number of correctly detected (predicted) rejected loan applications."*

**Primary Metric:** Recall for class 1 (Rejected), supported by AUC-ROC.

**Target Encoding:**
- 0 = Approved
- 1 = Rejected

## 1. Import Libraries
**Reused From:** Seminar 3 and Seminar 4 – classification libraries

In [ ]:
# Import pandas for data handling
import pandas as pd
# Import numpy for numerical operations
import numpy as np
# Import train_test_split for splitting data and GridSearchCV for tuning
from sklearn.model_selection import train_test_split, GridSearchCV
# Import classification algorithms
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
# Import evaluation metrics
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score,
                             recall_score, precision_score, f1_score,
                             roc_auc_score, RocCurveDisplay)
# Import standard scaler for feature scaling (required for LR and KNN)
from sklearn.preprocessing import StandardScaler
# Import matplotlib for plotting
import matplotlib.pyplot as plt
# Suppress non-critical warnings
import warnings
warnings.filterwarnings('ignore')
print("All libraries imported successfully.")

## 2. Mount Google Drive
**Reused From:** Seminar 1, Part (A) – mounting Google Drive

In [ ]:
from google.colab import drive
# Mount Google Drive to access the prepared classification dataset
drive.mount('/content/drive')

## 3. Load Classification Dataset
**Reused From:** Seminar 1, Part (C) – pd.read_csv()

In [ ]:
# Load the prepared classification dataset saved from Notebook 1
df = pd.read_csv('/content/drive/MyDrive/loan_classification_data.csv')
# Display shape and column names to verify correct load
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

## 4. Define Input Features (X) and Target (y)
**Reused From:** Seminar 3, Part (3) – Selecting Features; Seminar 4, Step 4 – assign inputs and target

In [ ]:
# Assign all columns except the target to X (input features)
X = df.drop(['loan_approval_status'], axis=1)
# Assign the target variable (loan approval status: 0=Approved, 1=Rejected)
y = df['loan_approval_status']
# Display feature names and shapes
print("Input features:", list(X.columns))
print("Input shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution (0=Approved, 1=Rejected):")
print(y.value_counts())
print(y.value_counts(normalize=True) * 100)

## 5. Train-Test Split
**Reused From:** Seminar 3, Part (1C) – train_test_split; Seminar 4, Step 5

**Split Ratio Justification:** A 75:25 split is used. With approximately 58,000 records, 25% (~14,500 instances) provides a statistically large enough test set to produce reliable performance estimates, while 75% (~43,500 instances) gives each model sufficient training data. Stratification (`stratify=y`) ensures the class ratio (Approved:Rejected) is preserved identically in both subsets, meeting the coursework requirement that all models are tested on the same instances with the same label distribution (Géron, 2022).

`random_state=42` ensures reproducibility — the same split is produced every run, so all three models are tested on identical test instances.

**Reference:** Géron, A. (2022) *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. 3rd ed. O'Reilly Media.

In [ ]:
# Split dataset into 75% training and 25% test sets
# random_state=42 ensures reproducibility (same split every run - all models tested on same instances)
# stratify=y preserves the Approved:Rejected class ratio in both training and test subsets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# Display shape of training and test sets
print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape: ", y_test.shape)

# Verify class stratification is preserved in training set
print("\nTraining class distribution (%):")
print(y_train.value_counts(normalize=True) * 100)
# Verify class stratification is preserved in test set
print("\nTest class distribution (%):")
print(y_test.value_counts(normalize=True) * 100)

## 6. Feature Scaling
**Reused From:** Seminar 3, Part (3) – Scale input variables with StandardScaler

Scaling is applied after the split to avoid data leakage — the scaler is fitted **only on the training data** and then applied to the test data.

In [ ]:
# Initialise StandardScaler to standardise input features (mean=0, std=1)
scaler = StandardScaler()
# Fit scaler on training data only and transform training set (prevents data leakage)
X_train_scaled = scaler.fit_transform(X_train)
# Transform test set using the training-fitted scaler (no re-fitting)
X_test_scaled = scaler.transform(X_test)
print("Features scaled with StandardScaler (fit on training set only to prevent data leakage).")

## 7. Build Model 1 – Naïve Bayes (NB)
**Reused From:** Seminar 4, Part (B) – GaussianNB; Seminar 3, Part (3) – fit and predict

**Algorithm Type:** Parametric | **Learnable Parameters:** Class priors (P(y)), feature likelihoods (mean and variance per class per feature) | **Strategic Hyperparameters:** `var_smoothing`

In [ ]:
# Instantiate the Gaussian Naive Bayes classifier
nb = GaussianNB()
# Fit (train) the NB model on the scaled training data
nb.fit(X_train_scaled, y_train)
# Predict loan approval status on the test data
y_pred_nb = nb.predict(X_test_scaled)
print("Naive Bayes model trained and tested.")

In [ ]:
# Construct and display the confusion matrix for Naive Bayes
nb_cm = confusion_matrix(y_test, y_pred_nb, labels=nb.classes_)
disp_nb = ConfusionMatrixDisplay(nb_cm, display_labels=['Approved(0)', 'Rejected(1)'])
disp_nb.plot(values_format='.0f')
plt.title('Confusion Matrix: Naive Bayes')
plt.show()

# Print the full classification report for Naive Bayes
print("Classification Report – Naive Bayes:")
print(classification_report(y_test, y_pred_nb, target_names=['Approved(0)', 'Rejected(1)']))

In [ ]:
# Plot the ROC curve for Naive Bayes
RocCurveDisplay.from_estimator(nb, X_test_scaled, y_test)
plt.title('ROC Curve: Naive Bayes')
plt.show()

## 8. Build Model 2 – Logistic Regression (LR)
**Reused From:** Seminar 3, Part (3) – LogisticRegression; Code Reuse Session 2

**Algorithm Type:** Parametric | **Learnable Parameters:** Weights (coefficients) and bias (intercept) | **Strategic Hyperparameters:** `C` (regularisation strength), `penalty` (L1/L2), `solver`

In [ ]:
# Instantiate the Logistic Regression classifier
# max_iter=1000 ensures convergence on this dataset; random_state=42 for reproducibility
lr = LogisticRegression(max_iter=1000, random_state=42)
# Fit (train) the LR model on the scaled training data
lr.fit(X_train_scaled, y_train)
# Predict loan approval status on the test data
y_pred_lr = lr.predict(X_test_scaled)
print("Logistic Regression model trained and tested.")

In [ ]:
# Construct and display the confusion matrix for Logistic Regression
lr_cm = confusion_matrix(y_test, y_pred_lr, labels=lr.classes_)
disp_lr = ConfusionMatrixDisplay(lr_cm, display_labels=['Approved(0)', 'Rejected(1)'])
disp_lr.plot(values_format='.0f')
plt.title('Confusion Matrix: Logistic Regression')
plt.show()

# Print the full classification report for Logistic Regression
print("Classification Report – Logistic Regression:")
print(classification_report(y_test, y_pred_lr, target_names=['Approved(0)', 'Rejected(1)']))

In [ ]:
# Plot the ROC curve for Logistic Regression
RocCurveDisplay.from_estimator(lr, X_test_scaled, y_test)
plt.title('ROC Curve: Logistic Regression')
plt.show()

## 9. Build Model 3 – K-Nearest Neighbours (KNN)
**Reused From:** Seminar 4, Part (A), Steps 6–9 – KNeighborsClassifier; error rate loop

**Algorithm Type:** Non-Parametric | **Learnable Parameters:** None (instance-based learner; no explicit training phase) | **Strategic Hyperparameters:** `n_neighbors` (k), `metric` (distance measure), `weights`

In [ ]:
# Find optimal k value by plotting error rate for k=1 to 40
error_rates = []
# Loop through k values from 1 to 40 and compute error rate for each
for k in range(1, 41):
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    knn_temp.fit(X_train_scaled, y_train)
    pred_temp = knn_temp.predict(X_test_scaled)
    # Calculate mean error rate (proportion of misclassified test instances)
    error_rates.append(np.mean(pred_temp != y_test))

# Plot the error rate curve to identify the elbow (optimal k)
plt.figure(figsize=(12, 6))
plt.plot(range(1, 41), error_rates, color='red', linestyle='dashed', marker='o',
         markerfacecolor='blue', markersize=8)
plt.title('KNN Error Rate vs K Value')
plt.xlabel('K Value')
plt.ylabel('Mean Error Rate')
plt.show()
print(f"Optimal k (lowest error rate): k={error_rates.index(min(error_rates)) + 1}")

In [ ]:
# Build KNN model with k=5 (selected as a balance between bias and variance)
knn = KNeighborsClassifier(n_neighbors=5)
# Train the KNN model on scaled training data
knn.fit(X_train_scaled, y_train)
# Predict loan approval status on the scaled test data
y_pred_knn = knn.predict(X_test_scaled)
print("KNN (k=5) model trained and tested.")

In [ ]:
# Construct and display the confusion matrix for KNN
knn_cm = confusion_matrix(y_test, y_pred_knn, labels=knn.classes_)
disp_knn = ConfusionMatrixDisplay(knn_cm, display_labels=['Approved(0)', 'Rejected(1)'])
disp_knn.plot(values_format='.0f')
plt.title('Confusion Matrix: KNN (k=5)')
plt.show()

# Print the full classification report for KNN
print("Classification Report – KNN (k=5):")
print(classification_report(y_test, y_pred_knn, target_names=['Approved(0)', 'Rejected(1)']))

In [ ]:
# Plot the ROC curve for KNN
RocCurveDisplay.from_estimator(knn, X_test_scaled, y_test)
plt.title('ROC Curve: KNN (k=5)')
plt.show()

## 10. Performance Comparison Summary
**Reused From:** Seminar 4, Step 8 – Evaluate the kNN model's test results

In [ ]:
# Build a summary table of all model test scores
# pos_label=1 targets the Rejected class (1=Rejected) per the coursework success criteria
models = {
    'NB': (nb, y_pred_nb),
    'LR': (lr, y_pred_lr),
    'KNN(k=5)': (knn, y_pred_knn)
}
results = []
for name, (model, y_pred) in models.items():
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        # Recall for Rejected class (pos_label=1): primary success metric
        'Recall (Reject=1)': recall_score(y_test, y_pred, pos_label=1),
        'Precision (Reject=1)': precision_score(y_test, y_pred, pos_label=1),
        'F1-Score (Reject=1)': f1_score(y_test, y_pred, pos_label=1),
        # AUC-ROC uses probability scores for the positive (Rejected) class
        'AUC-ROC': roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1])
    })

# Display results as a formatted table
results_df = pd.DataFrame(results).set_index('Model')
pd.set_option('display.float_format', '{:.4f}'.format)
print("Classification Performance Summary (Test Set):")
print(results_df)

## 11. Hyperparameter Tuning – Best Model with GridSearchCV
**Reused From:** Seminar 4, Step 10 – GridSearchCV for hyperparameter tuning

Based on Recall for class 1 (Rejected) being the primary success metric per the coursework brief, KNN is tuned using GridSearchCV to search for the best `n_neighbors` and `metric` combination. GridSearchCV uses 5-fold cross-validation scored on Recall to find the optimal hyperparameters.

In [ ]:
# Define the parameter grid to search for optimal KNN hyperparameters
param_grid = {
    'n_neighbors': np.arange(1, 25),      # Search k from 1 to 24
    'metric': ['euclidean', 'manhattan']   # Compare two common distance metrics
}
# Instantiate a new KNN classifier for tuning
knn_tune = KNeighborsClassifier()
# Apply GridSearchCV with 5-fold cross-validation
# Scoring on 'recall' targets maximising Rejected class detection (success criteria)
knn_gscv = GridSearchCV(knn_tune, param_grid, cv=5, scoring='recall')
# Fit the grid search on the scaled training data
knn_gscv.fit(X_train_scaled, y_train)
# Display the best hyperparameter combination found
print("Best hyperparameters:", knn_gscv.best_params_)
print("Best cross-validation recall:", knn_gscv.best_score_)

In [ ]:
# Predict with tuned KNN model on the test set
y_pred_knn_tuned = knn_gscv.predict(X_test_scaled)

# --- Confusion Matrix BEFORE tuning ---
print("--- BEFORE Tuning (KNN k=5) ---")
disp_before = ConfusionMatrixDisplay(knn_cm, display_labels=['Approved(0)', 'Rejected(1)'])
disp_before.plot(values_format='.0f')
plt.title('Confusion Matrix: KNN Before Tuning')
plt.show()

# --- Confusion Matrix AFTER tuning ---
print("--- AFTER Tuning (GridSearchCV) ---")
knn_tuned_cm = confusion_matrix(y_test, y_pred_knn_tuned, labels=knn_gscv.classes_)
disp_after = ConfusionMatrixDisplay(knn_tuned_cm, display_labels=['Approved(0)', 'Rejected(1)'])
disp_after.plot(values_format='.0f')
plt.title('Confusion Matrix: KNN After Tuning')
plt.show()

In [ ]:
# Print full classification reports before and after tuning
print("Classification Report - KNN BEFORE Tuning:")
print(classification_report(y_test, y_pred_knn, target_names=['Approved(0)', 'Rejected(1)']))

# Calculate key metrics before tuning
recall_before = recall_score(y_test, y_pred_knn, pos_label=1)
f1_before = f1_score(y_test, y_pred_knn, pos_label=1)
auc_before = roc_auc_score(y_test, knn.predict_proba(X_test_scaled)[:, 1])
print(f"Recall (Rejected=1) BEFORE tuning: {recall_before:.4f}")
print(f"F1-Score (Rejected=1) BEFORE tuning: {f1_before:.4f}")
print(f"AUC-ROC BEFORE tuning: {auc_before:.4f}")

print("\nClassification Report - KNN AFTER Tuning:")
print(classification_report(y_test, y_pred_knn_tuned, target_names=['Approved(0)', 'Rejected(1)']))

# Calculate key metrics after tuning
recall_after = recall_score(y_test, y_pred_knn_tuned, pos_label=1)
f1_after = f1_score(y_test, y_pred_knn_tuned, pos_label=1)
auc_after = roc_auc_score(y_test, knn_gscv.predict_proba(X_test_scaled)[:, 1])
print(f"Recall (Rejected=1) AFTER tuning: {recall_after:.4f}")
print(f"F1-Score (Rejected=1) AFTER tuning: {f1_after:.4f}")
print(f"AUC-ROC AFTER tuning: {auc_after:.4f}")

print(f"\nImprovement in Recall (Rejected): {recall_after - recall_before:+.4f}")